In [1]:
import traceback

import torch
from torch import nn
from torch.nn import functional as F

## Layers and Modules

In [2]:
net = nn.Sequential(
    nn.LazyLinear(256),
    nn.ReLU(),
    nn.LazyLinear(10)
)

X = torch.rand(2, 20)
net(X).shape

torch.Size([2, 10])

### Custom Module

In [3]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.hidden = nn.LazyLinear(256)
        self.out = nn.LazyLinear(10)

    def forward(self, X):
        return self.out(F.relu(self.hidden(X)))

In [4]:
net = MLP()

net(X).shape

torch.Size([2, 10])

### Sequential Module

In [5]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()

        for idx, module in enumerate(args):
            self.add_module(str(idx), module)

    def forward(self, X):
        for module in self.children():
            X = module(X)

        return X

In [6]:
net = MySequential(
    nn.LazyLinear(256),
    nn.ReLU(),
    nn.LazyLinear(10)
)

net(X).shape

torch.Size([2, 10])

### Nested Modules

In [7]:
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.LazyLinear(256),
            nn.ReLU(),
            nn.LazyLinear(10)
        )
        self.linear = nn.LazyLinear(16)

    def forward(self, X):
        return self.linear(self.net(X))

In [8]:
chimera = nn.Sequential(
    NestMLP(),
    nn.LazyLinear(20)
)

chimera(X).shape

torch.Size([2, 20])

## Parameter Management

In [9]:
net = nn.Sequential(
    nn.LazyLinear(8),
    nn.ReLU(),
    nn.LazyLinear(1)
)

net(X).shape

torch.Size([2, 1])

In [10]:
net[2].state_dict()

OrderedDict([('weight',
              tensor([[-0.1137, -0.1349,  0.1706,  0.1714,  0.2747, -0.1019,  0.2952,  0.2201]])),
             ('bias', tensor([0.0979]))])

In [11]:
type(net[2].bias), net[2].bias.data

(torch.nn.parameter.Parameter, tensor([0.0979]))

In [12]:
net[2].weight.grad == None

True

In [13]:
[(name, param.shape) for name, param in net.named_parameters()]

[('0.weight', torch.Size([8, 20])),
 ('0.bias', torch.Size([8])),
 ('2.weight', torch.Size([1, 8])),
 ('2.bias', torch.Size([1]))]

## Custom Layers

In [14]:
class CenteredLayer(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, X):
        return X - X.mean()

In [15]:
layer = CenteredLayer()

layer(torch.tensor([1.0, 2, 3, 4, 5]))

tensor([-2., -1.,  0.,  1.,  2.])

In [16]:
net = nn.Sequential(
    nn.LazyLinear(128),
    CenteredLayer()
)

Y = net(torch.rand(4, 8))
Y.mean()

tensor(4.6566e-10, grad_fn=<MeanBackward0>)

### Layers with Parameters

In [17]:
class MyLinear(nn.Module):
    def __init__(self, n_in, n_out):
        super().__init__()

        self.weight = nn.Parameter(torch.randn(n_in, n_out))
        self.bias = nn.Parameter(torch.randn(n_out))

    def forward(self, X):
        linear = X @ self.weight + self.bias
        
        return F.relu(linear)

In [18]:
linear = MyLinear(5, 3)

linear.weight

Parameter containing:
tensor([[-0.1023, -1.3919,  0.7036],
        [-0.3862, -0.0465,  2.6798],
        [-0.1794,  0.3249,  1.1274],
        [ 1.0199, -0.8136,  1.3668],
        [ 0.6604, -0.1832,  0.0033]], requires_grad=True)

In [19]:
linear(torch.rand(2, 5))

tensor([[0.0000, 0.0000, 3.8330],
        [0.0000, 0.0000, 2.8492]], grad_fn=<ReluBackward0>)

In [20]:
net = nn.Sequential(
    MyLinear(64, 8),
    MyLinear(8, 1)
)

net(torch.rand(2, 64))

tensor([[6.8842],
        [4.8222]], grad_fn=<ReluBackward0>)

## File I/O

### Loading and Saving Tensors

In [21]:
x = torch.arange(4)
x

tensor([0, 1, 2, 3])

In [22]:
torch.save(x, 'x-file')

In [23]:
x2 = torch.load('x-file')
x2

tensor([0, 1, 2, 3])

In [24]:
y = torch.zeros(4)
y

tensor([0., 0., 0., 0.])

In [25]:
torch.save([x, y], 'x-files')

In [26]:
x2, y2 = torch.load('x-files')
x2, y2

(tensor([0, 1, 2, 3]), tensor([0., 0., 0., 0.]))

In [27]:
mydict = {
    'x': x,
    'y': y
}

torch.save(mydict, 'mydict')

In [28]:
mydict2 = torch.load('mydict')

mydict2

{'x': tensor([0, 1, 2, 3]), 'y': tensor([0., 0., 0., 0.])}

### Loading and Saving Model Parameters

In [29]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.hidden = nn.LazyLinear(256)
        self.out = nn.LazyLinear(10)

    def forward(self, X):
        return self.out(F.relu(self.hidden(X)))

In [30]:
net = MLP()

X = torch.randn((2, 20))
Y = net(X)

In [31]:
torch.save(net.state_dict(), 'mlp.params')

In [32]:
clone = MLP()

clone.load_state_dict(torch.load('mlp.params'))
clone.eval()

MLP(
  (hidden): LazyLinear(in_features=0, out_features=256, bias=True)
  (out): LazyLinear(in_features=0, out_features=10, bias=True)
)

In [33]:
Y_clone = clone(X)

Y_clone == Y

tensor([[True, True, True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True]])

## GPUs

In [34]:
def cpu():
    return torch.device('cpu')

def gpu(i=0):
    return torch.device(f'cuda:{i}')

cpu(), gpu(), gpu(1)

(device(type='cpu'),
 device(type='cuda', index=0),
 device(type='cuda', index=1))

In [35]:
def num_gpus():
    return torch.cuda.device_count()

num_gpus()

1

In [36]:
def try_gpu(i=0):
    if num_gpus() >= i+1:
        return gpu(i)

    return cpu()

def try_all_gpus():
    return [gpu(i) for i in range(num_gpus())]

try_gpu(), try_gpu(10), try_all_gpus()

(device(type='cuda', index=0),
 device(type='cpu'),
 [device(type='cuda', index=0)])

### Tensors and GPU

In [37]:
X = torch.ones(2, 3)

X.device

device(type='cpu')

In [38]:
Y = torch.ones(2, 3, device=try_gpu())

Y

tensor([[1., 1., 1.],
        [1., 1., 1.]], device='cuda:0')

In [39]:
# supposed to fail

try:
    X + Y
except Exception as e:
    print('Type:', type(e).__name__)

    print('\tMessage:', str(e))
    print('\tRepresentation:', repr(e))
    print('\tArguments:', e.args)

    print('\nFormatted traceback:')
    print(traceback.format_exc())

Type: RuntimeError
	Message: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!
	Representation: RuntimeError('Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!')
	Arguments: ('Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!',)

Formatted traceback:
Traceback (most recent call last):
  File "/tmp/ipykernel_106349/2218080395.py", line 4, in <module>
    X + Y
    ~~^~~
RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!



In [40]:
Z = X.to(device=try_gpu())

In [41]:
Z + Y

tensor([[2., 2., 2.],
        [2., 2., 2.]], device='cuda:0')

### Neural Networks and GPU

In [42]:
net = nn.Sequential(nn.LazyLinear(1))

net = net.to(device=try_gpu())

In [43]:
# supposed to fail

try:
    net(X)
except Exception as e:
    print('Type:', type(e).__name__)

    print('\tMessage:', str(e))
    print('\tRepresentation:', repr(e))
    print('\tArguments:', e.args)

    print('\nFormatted traceback:')
    print(traceback.format_exc())

Type: RuntimeError
	Message: Expected all tensors to be on the same device, but got mat1 is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA_addmm)
	Representation: RuntimeError('Expected all tensors to be on the same device, but got mat1 is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA_addmm)')
	Arguments: ('Expected all tensors to be on the same device, but got mat1 is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA_addmm)',)

Formatted traceback:
Traceback (most recent call last):
  File "/tmp/ipykernel_106349/2471962546.py", line 4, in <module>
    net(X)
    ~~~^^^
  File "/home/leo/Projects/training/ai/ai_foundations/.venv/lib/python3.13/site-packages/torch/nn/modules/module.py", line 1783, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/home/leo/Projects/training/ai/ai_fou

In [44]:
net(Z)

tensor([[-0.3218],
        [-0.3218]], device='cuda:0', grad_fn=<AddmmBackward0>)

In [45]:
net[0].weight.data.device

device(type='cuda', index=0)